In [2]:
import os

In [3]:
%pwd

'd:\\Github Projects\\Pratice\\research'

In [4]:
import os

os.chdir(r"D:\Github Projects\Pratice")

print(os.getcwd())

D:\Github Projects\Pratice


In [5]:
%pwd

'D:\\Github Projects\\Pratice'

In [6]:
from pathlib import Path

print(Path.cwd())

D:\Github Projects\Pratice


In [7]:
import os
import pandas as pd

data = pd.read_csv("artifacts/data_ingestion/winequality-red.csv")
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


Update config entity

In [9]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DatavalidationConfig:
    root_dir: Path
    unzip_data_dir: Path
    STATUS_FILE: str
    all_schema: dict 


Update Configuration Manager

In [10]:
import sys
from pathlib import Path

from src.datascience.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from src.datascience.utils.common import read_yaml,create_directories
from src.datascience.entity.config_entity import DataIngestionConfig,DataValidationConfig

class ConfigurationManager:
    def __init__(self,config_filepath=CONFIG_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.data_validation.root_dir])

    def get_data_validation_config(self)->DataValidationConfig:
        config = self.config.data_validation
        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
                root_dir=config.root_dir,
                STATUS_FILE=config.STATUS_FILE,
                unzip_data_dir=config.unzip_data_dir,
                all_schema=config.all_schema
        )

        return data_validation_config


ImportError: cannot import name 'DataValidationConfig' from 'src.datascience.entity.config_entity' (D:\Github Projects\Pratice\src\datascience\entity\config_entity.py)

Update Components

In [ ]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self)->bool:
        try:
            validation_status = None

            data = pd.read_csv(self.config.unzip_data_dir)
            all_cols = list(data.columns)

            all_schema = self.config.all_schema.keys()

            for col in all_cols:
                if col not in all_schema:
                    validation_status = False
                    with open(self.config.validation_log_file, "w") as f:
                        f.write(f"Column '{col}' is in the data but not in the schema.\n")
                else:
                    validation_status = True
                    with open(self.config.validation_log_file, "a") as f:
                        f.write(f"Column '{col}' is in both the data and the schema.\n")
            return validation_status
        
        except Exception as e:
            print(f"Error occurred while validating columns: {e}")
            return False

NameError: name 'DataValidationConfig' is not defined

Pipeline

In [11]:
import os
from src.datascience.config.configuration import ConfigurationManager
from src.datascience.components.data_validation import DataValidation
from src.datascience import logger


STAGE_NAME = "Data Validation Stage"

class DataValidationTrainingPipeline:
    def __init__(self):
        pass

    def initiate_data_validation(self):
        config = ConfigurationManager()
        data_validation_config = config.get_data_validation_config()
        data_validation = DataValidation(config=data_validation_config)
        data_validation.validate_all_columns()



if __name__ == '__main__':
    try:
        logger.info(f"----{STAGE_NAME} started ----")
        pipeline = DataValidationTrainingPipeline()
        pipeline.initiate_data_validation()
        logger.info(f"----{STAGE_NAME} completed ----")
    except Exception as e:
        logger.error(f"Error occurred while initiating data validation: {e}")

ImportError: cannot import name 'DataValidationConfig' from 'src.datascience.entity.config_entity' (D:\Github Projects\Pratice\src\datascience\entity\config_entity.py)